In [3]:
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import (
    RunnableBranch,
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
    RunnableSequence,
)
from langchain_mistralai import ChatMistralAI

load_dotenv()

True

In [2]:
model= ChatMistralAI(model='mistral-medium-2508')

parser= StrOutputParser()

# Runnable Sequence

In [13]:
prompt1= PromptTemplate(
    template= 'tell me a joke about {topic}',
    input_variables= ['topic']
)

prompt2= PromptTemplate(
    template='Explain the following joke in short - {joke}',
    input_variables=['joke']
)

In [14]:
chain= RunnableSequence(prompt1, model, parser, prompt2, model, parser)
result= chain.invoke({'topic': 'chicken'})
print(result)

1. **First joke**: The chicken joined a band because it already has "drumsticks" (both the chicken legs and the drumming sticks).

2. **Bonus joke**: Chickens don’t tell jokes because they might "crack up" (both laughing uncontrollably and their eggs cracking).


# Runnable Parallel

In [16]:
prompt1= PromptTemplate(
    template='Generate a tweet about {topic}',
    input_variables=["topic"]
)

prompt2= PromptTemplate(
    template='Generate a linkedin post about {topic}',
    input_variables=["topic"]

)

In [19]:
parallel_chain= RunnableParallel({
    'tweet': RunnableSequence(prompt1, model, parser),
    'linkedin': RunnableSequence(prompt2, model, parser)
})

result= parallel_chain.invoke({'topic': 'War'})
print(result)

{'tweet': 'Here’s a punchy tweet about war—adjust the tone based on your intent (awareness, history, anti-war, etc.):\n\n---\n**"War doesn’t just kill soldiers—it starves children, erases homes, and buries futures. Every bomb dropped is a failure of humanity. When will we choose peace over power?"** #EndWar #PeaceNotProfit\n\n---\n**Alternate versions:**\n- *Historical angle:*\n  **"From Troy to Ukraine, war’s only constants: grief, lies, and the rich who never fight. History doesn’t repeat—it rhymes with blood."** #WarCrimes #NeverAgain\n\n- *Anti-war (direct):*\n  **"No war is just. Only the powerful call it ‘necessary.’ The rest of us pay in lives, limbs, and stolen years. Resist."** #NoMoreWar\n\n- *Philosophical:*\n  **"War is the ultimate admission that we’ve run out of words. And yet, we keep speaking in bullets."** #HumanityFirst\n\n- *Call to action:*\n  **"Your tax dollars fund wars. Your silence enables them. Demand accountability. **#DivestFromWar** #PeaceIsPossible"**', 'l

# Runnable Passthrough

In [ ]:
passthrough= RunnablePassthrough()
print(passthrough.invoke("test")) # this will print input directly without any processing

test


In [ ]:
prompt1= PromptTemplate(
    template= 'tell me a joke about {topic}',
    input_variables= ['topic']
)

prompt2= PromptTemplate(
    template='Explain the following joke in short - {joke}',
    input_variables=['joke']
)

In [26]:
joke_generator_chain= RunnableSequence(prompt1, model, parser)

parallel_chain= RunnableParallel({
    'joke': RunnablePassthrough(),
    'explanation': RunnableSequence(prompt2, model, parser)
})

final_chain= RunnableSequence(joke_generator_chain, parallel_chain)

result= final_chain.invoke({'topic': 'chicken'})
print(result)

{'joke': 'Here’s a fun and engaging tweet about chicken—pick your vibe!\n\n**🍗 Classic & Tasty:**\n*"Chicken: the ultimate protein that works for breakfast, lunch, dinner, AND midnight snacks. Grilled, fried, roasted, or in a wrap—it’s always a good idea. What’s your go-to chicken dish? 👇 #ChickenLovers"*\n\n**😂 Funny Take:**\n*"Me: I should eat healthier.\nAlso me: *sees fried chicken* … okay but just this once.\n*one hour later* … and again. #NoRegrets"*\n\n**🔥 Spicy Debate:**\n*"Hot take: Wings > tenders. Fight me. 🍗🔥 #ChickenWars"*\n\n**🌍 Global Flavors:**\n*"From Japanese karaage to Jamaican jerk, Nigerian pepper soup to American BBQ—chicken is the world’s most versatile protein. What’s your favorite way to eat it? 🌎✨"*\n\n**💡 Pro Tip:**\n*"PSA: Brining your chicken before cooking = game changer. Juicy, flavorful, and foolproof. You’re welcome. 🧂💦 #CookingHacks"*', 'explanation': 'Here’s a polished, engaging LinkedIn post inspired by your chicken tweet ideas—perfect for sparking c

# Runnable Lambda

In [29]:
def word_counter(text):
    return len(text.split())

runnable_word_counter= RunnableLambda(word_counter)
result= runnable_word_counter.invoke("hello world")
print(result)

2


In [30]:
prompt= PromptTemplate(
    template='Write a joke about {topic}',
    input_variables=['topic']
)

jokek_generation_chain= RunnableSequence(prompt, model, parser)

parallel_chain= RunnableParallel({
    'joke': RunnablePassthrough(),
    'word_counter': RunnableLambda(word_counter)
})

final_chain= RunnableSequence(jokek_generation_chain, parallel_chain)

result= final_chain.invoke({'topic': 'chicken'})
print(result)

{'joke': 'Here’s a classic with a twist:\n\n**Why did the chicken join a band?**\nBecause it had *drumsticks*!\n\n*(Bonus groan-worthy follow-up: And why did it get kicked out?)*\n*Because it kept *clucking* up the rhythm!* 🥁🐔', 'word_counter': 35}


In [31]:
print(result['joke'])
print(result['word_counter'])

Here’s a classic with a twist:

**Why did the chicken join a band?**
Because it had *drumsticks*!

*(Bonus groan-worthy follow-up: And why did it get kicked out?)*
*Because it kept *clucking* up the rhythm!* 🥁🐔
35


# Runnable Branch

In [5]:
def word_counter(text):
    return len(text.split())

In [10]:
prompt1= PromptTemplate(
    template="Write a detailed report on {topic}",
    input_variables=["topic"]
)

prompt2= PromptTemplate(
    template="Summarize the following text in under 300 words.\n{text}",
    input_variables=["text"]
)

report_generation_chain= RunnableSequence(prompt1, model, parser)

branch_chain= RunnableBranch(
    (lambda x: len(x.split())> 500, RunnableSequence(prompt2, model, parser)),
    RunnablePassthrough()
)

output_chain= RunnableSequence(report_generation_chain, branch_chain)

counter_chain= RunnableParallel({
    'report': RunnablePassthrough(),
    'word_counter': RunnableLambda(word_counter)
})

final_chain= RunnableSequence(output_chain, counter_chain)

result= final_chain.invoke({'topic': 'world war 2'})

print(result)

{'report': '### **Summary of World War II (1939–1945) – Under 300 Words**\n\nWorld War II, the deadliest conflict in history, involved **over 100 million people** from **30+ countries**, resulting in **70–85 million deaths**. It pitted the **Allies** (U.S., UK, USSR, China, France) against the **Axis** (Nazi Germany, Fascist Italy, Imperial Japan). Key causes included the **harsh Treaty of Versailles (1919)**, the **rise of fascism** (Hitler, Mussolini, militarist Japan), **appeasement policies**, and **economic depression**. The war began with **Germany’s invasion of Poland (1939)**, prompting Britain and France to declare war.\n\n**Major Theaters:**\n- **Europe:** Germany’s **blitzkrieg** conquered much of Europe (1939–1941), but failures at **Stalingrad (1942–43)** and **D-Day (1944)** led to its defeat. The **Holocaust** killed **6 million Jews** and **5 million others**.\n- **Pacific:** Japan’s **attack on Pearl Harbor (1941)** brought the U.S. into the war. Key battles like **Mid

In [11]:
print(result['report'])
print("-"*50)
print(result['word_counter'])

### **Summary of World War II (1939–1945) – Under 300 Words**

World War II, the deadliest conflict in history, involved **over 100 million people** from **30+ countries**, resulting in **70–85 million deaths**. It pitted the **Allies** (U.S., UK, USSR, China, France) against the **Axis** (Nazi Germany, Fascist Italy, Imperial Japan). Key causes included the **harsh Treaty of Versailles (1919)**, the **rise of fascism** (Hitler, Mussolini, militarist Japan), **appeasement policies**, and **economic depression**. The war began with **Germany’s invasion of Poland (1939)**, prompting Britain and France to declare war.

**Major Theaters:**
- **Europe:** Germany’s **blitzkrieg** conquered much of Europe (1939–1941), but failures at **Stalingrad (1942–43)** and **D-Day (1944)** led to its defeat. The **Holocaust** killed **6 million Jews** and **5 million others**.
- **Pacific:** Japan’s **attack on Pearl Harbor (1941)** brought the U.S. into the war. Key battles like **Midway (1942)** and *